In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import nltk 
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head(5)

In [ ]:
df.sample(10)

In [ ]:
df.info() # get the information of the data

In [ ]:
# change the ojbect to lower case and remove the sepcial characters
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review']=df['review'].apply(preprocess_text)
df.head(5)

In [ ]:
nltk.download('stopwords') # download the stopwords

stop_words = set(stopwords.words('english'))
keep_words = {'not','no',"don't","didn't","never"}

stop_words = stop_words - keep_words

In [ ]:
# Function to remove stopwords
def remove_stopwords(text):
    words = re.findall(r'\w+', text.lower())  # Tokenize text
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

# Apply stopword removal
df['review'] = df['review'].apply(remove_stopwords)

df.head()

In [ ]:
df['sentiment'].unique() # get the unique values of the sentiment

In [ ]:
# change the sentiment to 1 and 0
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

In [ ]:
# Apply TF-IDF vectorization with a limited feature size to reduce memory usage
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1,2)
)

X = vectorizer.fit_transform(df['review'])

In [ ]:
x=X
y=df['sentiment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Train a Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# Predict the sentiment of the test set
y_pred = clf.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: {:.2f}%".format(accuracy * 100))

# Classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix Display
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm)
disp.plot()
plt.show()

In [ ]:
# Train a Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

In [ ]:
def predict_sentiment(text):
    # preprocess
    text = preprocess_text(text)
    text = remove_stopwords(text)
    # vectorize
    text_vector = vectorizer.transform([text])
    # predict
    predicted = clf.predict(text_vector)
    return "Positive" if predicted == 1 else "Negative"

reviews = [
    "Amazing movie with a great storyline!",
    "I didn't enjoy this film at all.",
    "It was okay, nothing special."]

for review in reviews:
    print(f"Review: {review}\nPredicted Sentiment: {predict_sentiment(review)}")

In [ ]:
import pickle   
with open("random_forest_sentiment.pkl", "wb") as model_file:
    pickle.dump(clf, model_file)

print("Model saved to random_forest_sentiment.pkl successfully!")

# Save the trained vectorizer
with open("tfidf_vectorizer.pkl", "wb") as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)

print("Vectorizer saved to tfidf_vectorizer.pkl successfully!")